# Airbnb Nightly Price Prediction — Final Model

## Objective

Train and deploy the final Airbnb nightly price prediction model for Stockholm County.

The model uses six property features: room type, guest capacity, bedrooms, beds, district, and amenity count.

A Random Forest Regressor with logarithmic target transformation is trained using predefined hyperparameters established during previous experiments.

The complete preprocessing and regression pipeline is packaged with a SHAP explainer in one MLflow model. Calling the deployed model's `.predict()` returns a nightly price and a SHAP explanation for each listing.

This notebook focuses exclusively on final model training and deployment, without additional model selection or exploratory analysis.

In [1]:
%pip install shap

  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.4/544.4 kB 1.4 MB/s  0:00:00? eta -:--:--
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [shap]━━━━━━━━━━━━━ 2/3 [shap]

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.pyfunc

import json
from importlib.metadata import version
from tempfile import TemporaryDirectory
from pathlib import Path

import shap
from sqlalchemy import text

from sklearn.pipeline import Pipeline
from sklearn.compose import (
    ColumnTransformer,
    TransformedTargetRegressor
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from mlflow.models import infer_signature
from mlflow_utils import load_config, test_server_connection
from postgre_tools import Users, create_postgres_engine

# Reproducibility
RANDOM_STATE = 42

# Final feature configuration
FEATURES = [
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "district",
    "amenity_count"
]

NUMERIC_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "amenity_count"
]

CATEGORICAL_FEATURES = [
    "room_type",
    "district"
]

# Previously selected hyperparameters
RF_PARAMS = {
    "n_estimators": 352,
    "max_depth": 10,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "min_samples_split": 7,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

# MLflow configuration
EXPERIMENT_NAME = "airbnb-price-prediction"

print("=== FINAL MODEL CONFIGURATION ===")
print("Features:", FEATURES)
print("Numerical:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("Target transformation: log1p")
print("Algorithm: RandomForestRegressor")
print("MLflow experiment:", EXPERIMENT_NAME)

/Users/pettergustafsson/study/4DT911-proj-viz/git-repos/4dt911-vis-project-research/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== FINAL MODEL CONFIGURATION ===
Features: ['room_type', 'accommodates', 'bedrooms', 'beds', 'district', 'amenity_count']
Numerical: ['accommodates', 'bedrooms', 'beds', 'amenity_count']
Categorical: ['room_type', 'district']
Target transformation: log1p
Algorithm: RandomForestRegressor
MLflow experiment: airbnb-price-prediction


## 2. Data Retrieval

The Airbnb dataset originates from Inside Airbnb and contains property listings in Stockholm County.

Two PostgreSQL tables are used:

- **`listings`** contains nightly prices, property characteristics, room types, and district information.
- **`listing_amenities`** contains amenities associated with each listing.

The tables are linked using listing identifiers. Amenity count is calculated as the number of distinct amenities associated with each property.

Only listings with valid, positive nightly prices are included in the training dataset.

In [3]:

# Connect to PostgreSQL
engine = create_postgres_engine(Users.AIRBNB.load_user())

# Retrieve listings and amenities
listings_df = pd.read_sql_query(
    text("SELECT * FROM listings"),
    engine
)

listing_amenities_df = pd.read_sql_query(
    text("SELECT listing_id, amenity FROM listing_amenities"),
    engine
)

# Validate required columns
required_columns = {"id", "price", *FEATURES} - {"amenity_count"}

assert required_columns.issubset(listings_df.columns), (
    f"Missing listing columns: {required_columns - set(listings_df.columns)}"
)

assert {"listing_id", "amenity"}.issubset(
    listing_amenities_df.columns
)

assert listings_df["id"].is_unique

# Keep listings with valid prices
priced_df = listings_df.loc[
    listings_df["price"].notna()
    & (listings_df["price"] > 0)
].copy()

# Count distinct amenities per listing
amenity_counts = (
    listing_amenities_df
    .groupby("listing_id")["amenity"]
    .nunique()
    .rename("amenity_count")
)

# Merge datasets
model_df = priced_df.merge(
    amenity_counts,
    left_on="id",
    right_index=True,
    how="left",
    validate="one_to_one"
)

# Listings without amenity records receive a count of zero
model_df["amenity_count"] = (
    model_df["amenity_count"]
    .fillna(0)
    .astype(int)
)

# Prepare final training features and target
X = model_df[FEATURES].copy()
y = model_df["price"].copy()

print("=== TRAINING DATA ===")
print(f"Listings retrieved: {len(listings_df):,}")
print(f"Listings with valid prices: {len(model_df):,}")
print(f"Training observations: {len(X):,}")
print(f"Input features: {X.shape[1]}")

print("\n=== FEATURE TYPES ===")
print(X.dtypes.to_string())

print("\n=== DISTRICT INFORMATION ===")
print(f"Unique districts: {X['district'].nunique()}")
print(f"Missing districts: {X['district'].isna().sum():,}")

print("\nTraining dataset prepared successfully.")


=== TRAINING DATA ===
Listings retrieved: 4,806
Listings with valid prices: 3,264
Training observations: 3,264
Input features: 6

=== FEATURE TYPES ===
room_type            str
accommodates       int64
bedrooms         float64
beds             float64
district             str
amenity_count      int64

=== DISTRICT INFORMATION ===
Unique districts: 11
Missing districts: 0

Training dataset prepared successfully.


## 3. Final Model Training

The final model uses six predefined features and a Random Forest Regressor with hyperparameters established during previous experiments.

Numerical features are processed using median imputation. Categorical features are processed using most-frequent imputation and one-hot encoding.

The target variable is transformed using `log1p` during training and converted back to SEK using `expm1` during prediction.

The complete preprocessing and regression pipeline is trained on all 3,264 listings with valid nightly prices and will be saved to MLflow for deployment.

In [4]:

# === FINAL PREPROCESSING PIPELINE ===

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=10,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES)
    ],
    verbose_feature_names_out=False
)

# === RANDOM FOREST ===

regression_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(**RF_PARAMS))
])

# === LOGARITHMIC TARGET TRANSFORMATION ===

final_model = TransformedTargetRegressor(
    regressor=regression_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# === TRAIN FINAL MODEL ===

final_model.fit(X, y)

print("=== FINAL MODEL TRAINED ===")
print("Algorithm: RandomForestRegressor")
print("Target transformation: log1p")
print(f"Training observations: {len(X):,}")
print(f"Input features: {len(FEATURES)}")
print(f"Unique districts: {X['district'].nunique()}")

# Verify that the model produces valid price predictions
sample_predictions = final_model.predict(X.head(5))

assert np.isfinite(sample_predictions).all()
assert (sample_predictions > 0).all()

print("\n=== PREDICTION VERIFICATION ===")

for i, prediction in enumerate(sample_predictions, start=1):
    print(f"Listing {i}: {prediction:,.2f} SEK/night")

print("\nFinal model training completed successfully.")


=== FINAL MODEL TRAINED ===
Algorithm: RandomForestRegressor
Target transformation: log1p
Training observations: 3,264
Input features: 6
Unique districts: 11

=== PREDICTION VERIFICATION ===
Listing 1: 1,153.50 SEK/night
Listing 2: 1,309.33 SEK/night
Listing 3: 759.74 SEK/night
Listing 4: 857.52 SEK/night
Listing 5: 1,627.87 SEK/night

Final model training completed successfully.


## 4. Prepare the Model for MLflow

Serialize the trained scikit-learn pipeline using the same skops format. The local serialization is included as an artifact inside the final MLflow model along with the SHAP background data and configuration.

In [5]:
# === PREPARE MODEL METADATA ===

input_example = X.head(5).copy()

# Valid categorical values for frontend dropdowns
supported_categories = {
    "districts": sorted(
        X["district"].dropna().unique().tolist()
    ),
    "room_types": sorted(
        X["room_type"].dropna().unique().tolist()
    )
}

model_metadata = {
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "target": "price",
    "target_unit": "SEK_per_night",
    "target_transformation": "log1p",
    "training_observations": len(X),
    "supported_categories": supported_categories
}


# Keep the local package files together until MLflow has uploaded them.
artifact_workspace = TemporaryDirectory(prefix="airbnb-price-model-")
ARTIFACT_DIR = Path(artifact_workspace.name)
pipeline_path = ARTIFACT_DIR / "sklearn_pipeline"

mlflow.sklearn.save_model(
    sk_model=final_model,
    path=str(pipeline_path),
    serialization_format="skops",
    skops_trusted_types=[
        "numpy.dtype",
        "sklearn.tree._tree.Tree",
    ],
)

print("Local pipeline serialized successfully.")

Local pipeline serialized successfully.


## 5. Verify Pipeline Serialization

Reload the local pipeline and compare its predictions to the original fitted pipeline. The final section also verifies inference after loading the combined model from MLflow.

In [6]:
# === LOAD LOCAL SERIALIZED PIPELINE ===

loaded_model = mlflow.sklearn.load_model(
    str(pipeline_path)
)

# Use sample observations for verification
verification_input = X.head(10).copy()

# Predict using original and loaded models
original_predictions = final_model.predict(
    verification_input
)

loaded_predictions = loaded_model.predict(
    verification_input
)

# Compare predictions
prediction_difference = np.abs(
    original_predictions - loaded_predictions
)

print("=== PIPELINE SERIALIZATION VERIFICATION ===")
print("Observations tested:", len(verification_input))

print("\n=== PREDICTION COMPARISON ===")

comparison_df = pd.DataFrame({
    "Original": original_predictions,
    "Loaded": loaded_predictions,
    "Difference": prediction_difference
})

print(
    comparison_df
    .round(6)
    .to_string(index=False)
)

# Verify consistency
assert np.allclose(
    original_predictions,
    loaded_predictions,
    rtol=1e-10,
    atol=1e-8
), "Loaded model predictions differ!"

print("\nMaximum prediction difference:",
      prediction_difference.max())

print("\nModel verification successful!")

=== PIPELINE SERIALIZATION VERIFICATION ===
Observations tested: 10

=== PREDICTION COMPARISON ===
   Original      Loaded  Difference
1153.495490 1153.495490         0.0
1309.326489 1309.326489         0.0
 759.736621  759.736621         0.0
 857.521444  857.521444         0.0
1627.871991 1627.871991         0.0
 853.955544  853.955544         0.0
4241.839188 4241.839188         0.0
2104.841437 2104.841437         0.0
 558.795451  558.795451         0.0
 541.888303  541.888303         0.0

Maximum prediction difference: 1.8189894035458565e-12

Model verification successful!


## 6. SHAP Explainability

SHAP is used to explain how the six property characteristics contribute to the final model's predicted nightly price.

A model-agnostic permutation explainer is applied to the complete trained prediction pipeline, ensuring that feature contributions are expressed in SEK.

Categorical features (`room_type` and `district`) are encoded using reversible numerical mappings for SHAP calculations.

A reproducible background dataset is sampled from the training observations. The background data and categorical mappings are packaged with the trained pipeline. The deployed model rebuilds the explainer when it loads.

The deployed model calculates SHAP explanations for the listings passed to each prediction call.

In [7]:

# === SHAP CONFIGURATION ===

SHAP_BACKGROUND_SIZE = 30
SHAP_MAX_EVALS = 200

# Create mappings for the two categorical features
category_mappings = {
    feature: {
        category: code
        for code, category in enumerate(
            sorted(X[feature].dropna().unique())
        )
    }
    for feature in CATEGORICAL_FEATURES
}

reverse_category_mappings = {
    feature: {
        code: category
        for category, code in mapping.items()
    }
    for feature, mapping in category_mappings.items()
}


def encode_shap_input(df):
    """Encode categorical inputs for SHAP."""

    encoded = df[FEATURES].copy()

    for feature, mapping in category_mappings.items():
        encoded[feature] = encoded[feature].map(mapping)

    return encoded.astype(float)


def predict_for_shap(values):
    """Convert SHAP inputs into model predictions in SEK."""

    df = pd.DataFrame(
        values,
        columns=FEATURES
    ).copy()

    for feature, mapping in reverse_category_mappings.items():
        df[feature] = [
            mapping.get(int(round(code))) if pd.notna(code)
            else np.nan
            for code in df[feature]
        ]

    return final_model.predict(df[FEATURES])


# Reproducible background from training data
shap_background = encode_shap_input(
    X.sample(
        n=min(SHAP_BACKGROUND_SIZE, len(X)),
        random_state=RANDOM_STATE
    )
)

# Verify that encoding does not change predictions
sample = X.sample(n=5, random_state=123)
encoded_sample = encode_shap_input(sample)

original_predictions = final_model.predict(sample)
wrapped_predictions = predict_for_shap(
    encoded_sample.to_numpy()
)

assert np.allclose(
    original_predictions,
    wrapped_predictions,
    rtol=1e-10,
    atol=1e-8
)

print("=== SHAP CONFIGURATION ===")
print("Features:", FEATURES)
print("Background observations:", len(shap_background))
print("Categorical mappings:")

for feature, mapping in category_mappings.items():
    print(f"  {feature}: {len(mapping)} categories")

print("\nPrediction consistency verified.")


=== SHAP CONFIGURATION ===
Features: ['room_type', 'accommodates', 'bedrooms', 'beds', 'district', 'amenity_count']
Background observations: 30
Categorical mappings:
  room_type: 3 categories
  district: 11 categories

Prediction consistency verified.


In [8]:

# === INITIALIZE SHAP EXPLAINER ===

shap_explainer = shap.Explainer(
    predict_for_shap,
    shap_background,
    algorithm="permutation",
    seed=RANDOM_STATE
)

# Select one listing for verification
shap_test_listing = X.sample(
    n=1,
    random_state=123
)

shap_test_encoded = encode_shap_input(
    shap_test_listing
)

# Calculate SHAP contributions in SEK
shap_test_values = shap_explainer(
    shap_test_encoded,
    max_evals=SHAP_MAX_EVALS
)

# Verify explanation reconstruction
predicted_price = float(
    final_model.predict(shap_test_listing)[0]
)

shap_base_value = float(
    shap_test_values.base_values[0]
)

shap_contributions = shap_test_values.values[0]

reconstructed_price = (
    shap_base_value + shap_contributions.sum()
)

reconstruction_difference = abs(
    predicted_price - reconstructed_price
)

print("=== SHAP VERIFICATION ===")
print(f"Predicted price: {predicted_price:.2f} SEK")
print(f"SHAP baseline: {shap_base_value:.2f} SEK")
print(f"Reconstructed price: {reconstructed_price:.2f} SEK")
print(f"Difference: {reconstruction_difference:.6f} SEK")

print("\n=== FEATURE CONTRIBUTIONS ===")

contributions_df = pd.DataFrame({
    "feature": FEATURES,
    "shap_sek": shap_contributions
})

print(
    contributions_df
    .round(2)
    .to_string(index=False)
)

assert shap_test_values.values.shape == (1, len(FEATURES))
assert reconstruction_difference < 1.0, (
    "SHAP reconstruction differs by at least 1 SEK"
)

print("\nSHAP verification successful!")


=== SHAP VERIFICATION ===
Predicted price: 542.85 SEK
SHAP baseline: 2063.68 SEK
Reconstructed price: 542.85 SEK
Difference: 0.000000 SEK

=== FEATURE CONTRIBUTIONS ===
      feature  shap_sek
    room_type   -364.68
 accommodates   -499.92
     bedrooms   -216.01
         beds    -39.17
     district   -398.88
amenity_count     -2.17

SHAP verification successful!


In [9]:
# === PREPARE SHAP ARTIFACTS ===

background_path = ARTIFACT_DIR / "shap_background.csv"
config_path = ARTIFACT_DIR / "shap_config.json"

# Save the exact background observations used by SHAP
shap_background.to_csv(
    background_path,
    index=False
)

# Preserve everything needed to reconstruct the explainer
shap_config = {
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "category_mappings": {
        feature: {
            str(category): int(code)
            for category, code in mapping.items()
        }
        for feature, mapping in category_mappings.items()
    },
    "algorithm": "permutation",
    "max_evals": SHAP_MAX_EVALS,
    "background_samples": len(shap_background),
    "random_state": RANDOM_STATE,
    "output_unit": "SEK",
    "background_file": "shap_background.csv"
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        shap_config,
        f,
        indent=2,
        ensure_ascii=False
    )

# === VALIDATE SAVED ARTIFACTS ===

saved_background = pd.read_csv(background_path)

with open(config_path, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

assert saved_background.shape == shap_background.shape
assert saved_config["features"] == FEATURES
assert saved_config["background_samples"] == len(saved_background)

## 7. Log and Verify the Explainable Model

The MLflow Python model packages the fitted pipeline, the SHAP background and configuration, and the code that combines them. On load, it reads the bundled artifacts and initializes the explainer. Each call to `predict()` returns one JSON-serializable dictionary per input row, in input order.

For each listing, the baseline plus the six feature contributions reconstructs the predicted nightly price in SEK. These contributions describe the model's prediction relative to its background listings.

In [11]:
# === PACKAGE AND LOG ONE DEPLOYABLE MODEL ===

wrapper_source = '''
import json
from pathlib import Path
from threading import Lock

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import shap


class ExplainableAirbnbPriceModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        with Path(context.artifacts["shap_config"]).open(encoding="utf-8") as file:
            config = json.load(file)

        self.pipeline = mlflow.sklearn.load_model(context.artifacts["pipeline"])
        self.features = config["features"]
        self.category_mappings = config["category_mappings"]
        self.max_evals = config["max_evals"]
        self.reverse_mappings = {}

        for feature, mapping in self.category_mappings.items():
            # The fitted encoder handles unseen labels as one group.
            unknown_label = "__unseen_category__"
            while unknown_label in mapping:
                unknown_label += "_"
            reverse_mapping = {code: label for label, code in mapping.items()}
            reverse_mapping[-1] = unknown_label
            self.reverse_mappings[feature] = reverse_mapping

        background = pd.read_csv(context.artifacts["shap_background"])
        self.explainer = shap.Explainer(
            self._predict_for_shap,
            background[self.features],
            algorithm=config["algorithm"],
            seed=config["random_state"],
        )
        self._explainer_lock = Lock()

    def _encode_for_shap(self, rows):
        encoded = rows.copy()
        for feature, mapping in self.category_mappings.items():
            values = rows[feature]
            codes = values.map(mapping)
            # Keep unseen labels distinct from genuinely missing values.
            encoded[feature] = codes.mask(values.notna() & codes.isna(), -1)
        return encoded.astype(float)

    def _predict_for_shap(self, values):
        rows = pd.DataFrame(values, columns=self.features).copy()
        for feature, reverse_mapping in self.reverse_mappings.items():
            rows[feature] = [
                reverse_mapping[int(code)] if pd.notna(code) else np.nan
                for code in rows[feature]
            ]
        return self.pipeline.predict(rows)

    def predict(self, context, model_input):
        rows = model_input[self.features].copy()
        for feature in self.category_mappings:
            rows[feature] = rows[feature].where(rows[feature].notna(), np.nan)

        prices = self.pipeline.predict(rows)
        encoded = self._encode_for_shap(rows)
        with self._explainer_lock:
            explanations = self.explainer(encoded, max_evals=self.max_evals)

        return [
            {
                "predicted_price_sek": float(price),
                "explanation": {
                    "baseline_sek": float(baseline),
                    "contributions_sek": {
                        feature: float(contribution)
                        for feature, contribution in zip(self.features, contributions)
                    },
                },
            }
            for price, baseline, contributions in zip(
                prices, explanations.base_values, explanations.values
            )
        ]


mlflow.models.set_model(ExplainableAirbnbPriceModel())
'''

wrapper_path = ARTIFACT_DIR / "explainable_airbnb_model.py"
wrapper_path.write_text(wrapper_source, encoding="utf-8")

artifacts = {
    "pipeline": str(pipeline_path),
    "shap_config": str(config_path),
    "shap_background": str(background_path),
}

# Use a prediction from the earlier SHAP verification to define the output schema.
output_example = [{
    "predicted_price_sek": predicted_price,
    "explanation": {
        "baseline_sek": shap_base_value,
        "contributions_sek": dict(zip(FEATURES, shap_contributions.tolist())),
    },
}]
signature = infer_signature(input_example, output_example)

# Record the versions used to train and package the model.
pip_requirements = [
    f"{package}=={version(package)}"
    for package in (
        "mlflow-skinny", "numpy", "pandas", "scipy", "scikit-learn", "shap", "skops"
    )
]

load_config()
test_server_connection()
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="airbnb-final-six-feature-model") as run:
    mlflow.log_params({
        "algorithm": "RandomForestRegressor",
        "feature_count": len(FEATURES),
        "features": FEATURES,
        "training_observations": len(X),
        "target_transformation": "log1p",
        "shap_algorithm": shap_config["algorithm"],
        "shap_max_evals": SHAP_MAX_EVALS,
        **RF_PARAMS,
    })
    mlflow.log_dict(model_metadata, "model_metadata.json")
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=str(wrapper_path),
        artifacts=artifacts,
        signature=signature,
        input_example=input_example.iloc[[0]],
        pip_requirements=pip_requirements,
    )
    final_run_id = run.info.run_id
    final_model_uri = model_info.model_uri

print("Experiment:", EXPERIMENT_NAME)
print("Run ID:", final_run_id)
print("Model URI:", final_model_uri)

Configuration loaded successfully.
MLflow connection successful!
Found 1 experiment(s)


/Users/pettergustafsson/study/4DT911-proj-viz/git-repos/4dt911-vis-project-research/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/11/09 08:37:11 INFO mlflow.pyfunc: Validating input example against model signature
/Users/pettergustafsson/study/4DT911-proj-viz/git-repos/4dt911-vis-project-research/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


🏃 View run airbnb-final-six-feature-model at: https://cu0089.camp.lnu.se/mlflow/#/experiments/2/runs/a958ed4ed9d14c6586c19d6af02952d7
🧪 View experiment at: https://cu0089.camp.lnu.se/mlflow/#/experiments/2
Experiment: airbnb-price-prediction
Run ID: a958ed4ed9d14c6586c19d6af02952d7
Model URI: models:/m-ddd79ffe3c8a4323bf3ae8440818383c


In [12]:
# Load the combined package and verify a batch in a known row order.
deployed_model = mlflow.pyfunc.load_model(final_model_uri)
verification_input = X.sample(n=min(3, len(X)), random_state=RANDOM_STATE)
deployed_results = deployed_model.predict(verification_input)
expected_prices = final_model.predict(verification_input)

assert len(deployed_results) == len(verification_input)
for result, expected_price in zip(deployed_results, expected_prices):
    price = result["predicted_price_sek"]
    explanation = result["explanation"]
    contributions = explanation["contributions_sek"]

    assert list(contributions) == FEATURES
    assert np.isclose(price, expected_price, rtol=1e-10, atol=1e-8)
    reconstructed_price = explanation["baseline_sek"] + sum(contributions.values())
    assert np.isclose(price, reconstructed_price, rtol=0.0, atol=1e-6)

# Strict JSON serialization also rejects NaN and infinity.
assert json.loads(json.dumps(deployed_results, allow_nan=False)) == deployed_results
print("Verified model reload, batch row order, SHAP reconstruction, and JSON output.")

/Users/pettergustafsson/study/4DT911-proj-viz/git-repos/4dt911-vis-project-research/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Verified model reload, batch row order, SHAP reconstruction, and JSON output.


## 8. Example. Predict from MLflow

Use the model URI printed above. In a separate application, configure MLflow access as usual and substitute that URI. The packaged model loads its own SHAP artifacts and only needs the six listing features as input.

`predict()` returns native Python dictionaries in a list, with one dictionary per listing. FastAPI can serialize the list directly. Use `json.dumps()` when you need a JSON string.

In [13]:
# In a separate application, replace final_model_uri with the logged model URI.
model = mlflow.pyfunc.load_model(final_model_uri)

property_data = pd.DataFrame([{
    "room_type": "Entire home/apt",
    "accommodates": 4,
    "bedrooms": 2.0,
    "beds": 3.0,
    "district": "Södermalm",
    "amenity_count": 20,
}])

results = model.predict(property_data)
print(json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False))

/Users/pettergustafsson/study/4DT911-proj-viz/git-repos/4dt911-vis-project-research/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


[
  {
    "predicted_price_sek": 3047.3624703391124,
    "explanation": {
      "baseline_sek": 2063.6834935080365,
      "contributions_sek": {
        "room_type": 211.18922045485203,
        "accommodates": 162.56926125822375,
        "bedrooms": 63.93963935727481,
        "beds": 28.484065824435206,
        "district": 601.0882318382426,
        "amenity_count": -83.59144190194777
      }
    }
  }
]
